## 1. 기본 패키지 불러오기
전처리와 리포트 저장에 필요한 패키지를 불러옵니다. 실제 경로와 처리 로직은 기존 코드 그대로 유지했습니다.

In [1]:
# 기본 패키지
from pathlib import Path
from datetime import datetime
import json
import os
import warnings

import polars as pl
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

print("polars:", pl.__version__)
print("pandas:", pd.__version__)


polars: 1.35.2
pandas: 2.2.2


## 2. 팀 공통 Google Drive 경로 설정
원본 데이터 위치와 저장 폴더를 지정합니다. Colab 환경에서 바로 실행되도록 기존 경로를 유지합니다.

In [2]:
from pathlib import Path

# ============================================================
# Google Drive 경로 설정
# ============================================================
# 팀원 공통 실행 기준:
# - raw 원본 데이터: 기업 Google Drive
# - 큰 parquet / 팀 공유 산출물: 기업 Google Drive
#
# Drive 폴더 구조:
# Graph_AML/
# └── data/
#     ├── raw/
#     ├── processed/
#     │   ├── clean_base/
#     │   └── ml_features/
#     │   └── gnn_inputs/
#
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    raise RuntimeError(
        "이 노트북은 기업용 Colab + Google Drive 실행 기준입니다. "
        "Colab에서 열고 Google Drive를 마운트한 뒤 실행해주세요."
    )

# 모든 산출물은 Google Drive에 저장합니다.
DRIVE_BASE_DIR = Path("/content/drive/MyDrive/Graph_AML")
DRIVE_DATA_DIR = DRIVE_BASE_DIR / "data"

RAW_DIR = DRIVE_DATA_DIR / "raw"
PROCESSED_DIR = DRIVE_DATA_DIR / "processed"

CLEAN_BASE_DIR = PROCESSED_DIR / "clean_base"
ML_DRIVE_DIR = PROCESSED_DIR / "ml_features"
GNN_DRIVE_DIR = PROCESSED_DIR / "gnn_inputs"
VALIDATION_DIR = CLEAN_BASE_DIR / "validation"


# 기존 코드 호환용 alias
BASE_DIR = DRIVE_BASE_DIR
DATA_DIR = DRIVE_DATA_DIR
CLEAN_BASE_DIR = CLEAN_BASE_DIR
ML_DIR = ML_DRIVE_DIR
GNN_DIR = GNN_DRIVE_DIR
PROJECT_DIR = DRIVE_BASE_DIR

for p in [
    RAW_DIR,
    PROCESSED_DIR,
    CLEAN_BASE_DIR,
    ML_DRIVE_DIR,
    GNN_DRIVE_DIR,
    VALIDATION_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

print("DRIVE_BASE_DIR   :", DRIVE_BASE_DIR)
print("DRIVE_DATA_DIR   :", DRIVE_DATA_DIR)
print("RAW_DIR          :", RAW_DIR)
print("CLEAN_BASE_DIR   :", CLEAN_BASE_DIR)
print("VALIDATION_DIR   :", VALIDATION_DIR)
print("ML_DRIVE_DIR     :", ML_DRIVE_DIR)
print("GNN_DRIVE_DIR    :", GNN_DRIVE_DIR)


# 02번은 raw 거래 파일을 읽고 clean base를 저장합니다.
def resolve_raw_transaction_path(raw_dir):
    raw_dir = Path(raw_dir)
    candidate_names = [
        "HI-Small_Trans.csv", "HI-Small_Trans.parquet",
        "LI-Small_Trans.csv", "LI-Small_Trans.parquet",
        "HI-Medium_Trans.csv", "LI-Medium_Trans.csv",
        "HI-Large_Trans.csv", "LI-Large_Trans.csv",
        "transactions.csv", "Transactions.csv", "trans.csv",
    ]
    for name in candidate_names:
        p = raw_dir / name
        if p.exists():
            return p

    files = sorted([p for p in raw_dir.iterdir() if p.is_file()])
    trans_like = [
        p for p in files
        if p.suffix.lower() in [".csv", ".parquet", ".pq"]
        and ("trans" in p.name.lower() or "transaction" in p.name.lower())
    ]
    if len(trans_like) == 1:
        return trans_like[0]
    if len(trans_like) > 1:
        print("거래 파일 후보가 여러 개입니다. 첫 번째 파일을 사용합니다:")
        for p in trans_like:
            print(" -", p.name)
        return trans_like[0]

    available = "\n".join([f" - {p.name}" for p in files[:50]])
    raise FileNotFoundError(
        "거래 데이터 파일을 자동으로 찾지 못했습니다.\n"
        "Graph_AML/data/raw 폴더의 파일명을 확인해주세요.\n"
        f"RAW_DIR: {raw_dir}\n확인된 파일:\n{available}"
    )

RAW_DATA_PATH = resolve_raw_transaction_path(RAW_DIR)
DRIVE_OUTPUT_DIR = CLEAN_BASE_DIR
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "BASE_DIR": BASE_DIR,
    "DATA_DIR": DATA_DIR,
    "RAW_DATA_PATH": RAW_DATA_PATH,
    "ACCOUNTS_PATH": RAW_DIR / "HI-Small_accounts.csv",       # 참고용. 이번 단계에서는 읽지 않음
    "PATTERNS_PATH": RAW_DIR / "HI-Small_Patterns.txt",       # 참고용. 피처로 사용 금지

    # 큰 산출물 / 다음 단계 입력용은 Drive에 저장
    "DRIVE_OUTPUT_DIR": DRIVE_OUTPUT_DIR,

    "LOCAL_OUTPUT_DIR": DRIVE_OUTPUT_DIR,

    # 기존 코드 호환용: 02번의 주 산출물 위치는 Drive
    "OUTPUT_DIR": DRIVE_OUTPUT_DIR,
    "VALIDATION_DIR": VALIDATION_DIR,
    "SAMPLE_MODE": False,
    "SAMPLE_ROWS": 100_000,
    "INFER_SCHEMA_LENGTH": 10000,

    # 은행 ID + 계좌번호를 묶어서 계좌 ID로 사용
    # 원본 Account 값만 쓰려면 False로 변경
    "USE_BANK_ACCOUNT_COMPOSITE_ID": True,

    "TIMESTAMP_FORMATS": [
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d %H:%M",
        "%Y-%m-%d %H:%M:%S%.f",
        "%Y/%m/%d %H:%M:%S",
        "%Y/%m/%d %H:%M",
    ],

    "TIMESTAMP_COL_CANDIDATES": [
        "Timestamp", "timestamp", "Time", "time", "DateTime", "datetime"
    ],
    "LABEL_COL_CANDIDATES": [
        "Is Laundering", "Is_Laundering", "is_laundering", "label", "Label", "target"
    ],
    "SENDER_COL_CANDIDATES": [
        "Account", "From Account", "From_Account", "from_account", "sender_account"
    ],
    "RECEIVER_COL_CANDIDATES": [
        "Account.1", "To Account", "To_Account", "to_account", "receiver_account"
    ],
    "FROM_BANK_COL_CANDIDATES": [
        "From Bank", "From_Bank", "from_bank"
    ],
    "TO_BANK_COL_CANDIDATES": [
        "To Bank", "To_Bank", "to_bank"
    ],
    "AMOUNT_COL_CANDIDATES": [
        "Amount Paid", "Amount_Paid", "amount_paid", "Amount Received", "Amount_Received", "amount"
    ],
    "CURRENCY_COL_CANDIDATES": [
        "Payment Currency", "Payment_Currency", "Receiving Currency", "Receiving_Currency", "Currency"
    ],
    "PAYMENT_FORMAT_COL_CANDIDATES": [
        "Payment Format", "Payment_Format", "payment_format", "Payment Type"
    ],
    "TX_ID_COL_CANDIDATES": [
        "tx_id", "transaction_id", "Transaction ID", "Transaction_ID", "id", "ID"
    ],

    "STRICT_LABEL_CHECK": True,
    "RANDOM_SEED": 42,

    # ============================================================
    # Primary period filter / split policy
    # ============================================================
    # Kaggle data card 기준 HI/LI 2022 primary period는 2022-09-01 ~ 2022-09-10
    # 원본 CSV는 물리적으로 수정하지 않고, 전처리 단계에서 primary period 밖 tail만 제외.
    # row count(1108건) 기준으로 자르지 않고 timestamp cutoff 기준으로만 필터링.
    "APPLY_PRIMARY_PERIOD_FILTER": True,
    "PRIMARY_PERIOD_START_INCLUSIVE": "2022-09-01 00:00:00",
    "PRIMARY_PERIOD_END_EXCLUSIVE": "2022-09-11 00:00:00",
    "PRIMARY_PERIOD_NOTE": "HI/LI 2022 Kaggle primary period: 2022-09-01 through 2022-09-10. Keep timestamp >= start and timestamp < end_exclusive.",

    # primary period 적용 후 시간순 60/20/20 split을 재생성합니다.
    "SPLIT_RATIOS": {"train": 0.60, "val": 0.20, "test": 0.20},
    "EXPECTED_SPLIT_POSITIVE_RATE": {
        "train": 0.000754,  # 0.0754%
        "val": 0.001066,    # 0.1066%
        "test": 0.001126,   # 0.1126%
    },
    # 기대 라벨 비율과의 허용 차이. 0.00005 = 0.005 percentage point.
    "EXPECTED_RATE_TOLERANCE": 0.00005,

    # polars 0.20.31 기준.
    "SCHEMA_OVERRIDES": {
        "Timestamp": pl.Utf8,
        "From Bank": pl.Utf8,
        "To Bank": pl.Utf8,
        "Account": pl.Utf8,
        "Account.1": pl.Utf8,
        "Amount Received": pl.Float64,
        "Amount Paid": pl.Float64,
        "Receiving Currency": pl.Utf8,
        "Payment Currency": pl.Utf8,
        "Payment Format": pl.Utf8,
        "Is Laundering": pl.Int64,
    },
}

print("BASE_DIR:", CONFIG["BASE_DIR"])
print("DATA_DIR:", CONFIG["DATA_DIR"])
print("RAW     :", CONFIG["RAW_DATA_PATH"], "exists=", CONFIG["RAW_DATA_PATH"].exists())
print("DRIVE_OUT:", CONFIG["DRIVE_OUTPUT_DIR"])

Mounted at /content/drive
DRIVE_BASE_DIR   : /content/drive/MyDrive/Graph_AML
DRIVE_DATA_DIR   : /content/drive/MyDrive/Graph_AML/data
RAW_DIR          : /content/drive/MyDrive/Graph_AML/data/raw
CLEAN_BASE_DIR   : /content/drive/MyDrive/Graph_AML/data/processed/clean_base
VALIDATION_DIR   : /content/drive/MyDrive/Graph_AML/data/processed/clean_base/validation
ML_DRIVE_DIR     : /content/drive/MyDrive/Graph_AML/data/processed/ml_features
GNN_DRIVE_DIR    : /content/drive/MyDrive/Graph_AML/data/processed/gnn_inputs
BASE_DIR: /content/drive/MyDrive/Graph_AML
DATA_DIR: /content/drive/MyDrive/Graph_AML/data
RAW     : /content/drive/MyDrive/Graph_AML/data/raw/HI-Small_Trans.csv exists= True
DRIVE_OUT: /content/drive/MyDrive/Graph_AML/data/processed/clean_base


## 3. 공통 유틸 함수 준비
검증 결과, 메모리 사용량, 저장 리포트를 만들 때 반복해서 쓰는 함수입니다. 코드 중복을 줄이고, 문제가 생겼을 때 어디서 발생했는지 추적하기 쉽게 합니다.

In [3]:
# 저장과 검증에 반복해서 쓰는 함수들
# =========================
# 유틸
# =========================

validation_records = []
memory_records = []


def _now():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def add_validation(check_name, status, severity, details, n_failed_rows=0):
    validation_records.append({
        "record_type": "validation",
        "check_name": check_name,
        "status": status,
        "severity": severity,
        "details": str(details),
        "n_failed_rows": int(n_failed_rows or 0),
        "created_at": _now(),
    })


def normalize_name(x):
    return str(x).strip().lower().replace("_", " ").replace("-", " ")


def detect_column(columns, candidates, required=True, logical_name="column"):
    norm_to_real = {normalize_name(c): c for c in columns}
    for cand in candidates:
        key = normalize_name(cand)
        if key in norm_to_real:
            return norm_to_real[key]

    if required:
        raise ValueError(
            f"필수 컬럼을 찾지 못했습니다: {logical_name}\n"
            f"후보: {candidates}\n"
            f"현재 컬럼: {list(columns)}"
        )
    return None


def estimate_memory_mb(df):
    try:
        return float(df.estimated_size("mb"))
    except Exception:
        return None


def profile_memory(df, step_name):
    memory_mb = estimate_memory_mb(df)
    memory_records.append({
        "step_name": step_name,
        "n_rows": df.height,
        "n_cols": df.width,
        "memory_mb": memory_mb,
        "created_at": _now(),
    })
    msg = f"[{step_name}] rows={df.height:,}, cols={df.width:,}"
    if memory_mb is not None:
        msg += f", memory={memory_mb:.2f} MB"
    print(msg)


# 한글이 깨지지 않도록 utf-8-sig로 CSV를 저장합니다.
def write_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.write_csv(path)


def compact_col_name(name):
    return str(name).strip().lower().replace(" ", "_").replace(".", "_").replace("-", "_")


## 4. 원본 거래 데이터 로드
원본 CSV를 읽어 공통 clean base를 만들 준비를 합니다. 이 단계부터는 EDA가 아니라 실제 전처리 파이프라인입니다.

In [4]:
# 원본 거래 데이터를 읽는 단계
# =========================
# 데이터 로드
# =========================

# 원본 거래 파일을 읽고, 이후 단계에서 필요한 기본 DataFrame을 준비합니다.
def read_input_data(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"입력 파일이 없습니다: {path}")

    suffix = path.suffix.lower()
    n_rows = CONFIG["SAMPLE_ROWS"] if CONFIG["SAMPLE_MODE"] else None

    if suffix == ".csv":
        # polars 버전별 인자명 차이 대응
        try:
            df = pl.read_csv(
                path,
                infer_schema_length=CONFIG["INFER_SCHEMA_LENGTH"],
                schema_overrides=CONFIG["SCHEMA_OVERRIDES"],
                n_rows=n_rows,
                ignore_errors=False,
            )
        except TypeError:
            df = pl.read_csv(
                path,
                infer_schema_length=CONFIG["INFER_SCHEMA_LENGTH"],
                dtypes=CONFIG["SCHEMA_OVERRIDES"],
                n_rows=n_rows,
                ignore_errors=False,
            )
    elif suffix in [".parquet", ".pq"]:
        if CONFIG["SAMPLE_MODE"]:
            df = pl.scan_parquet(path).limit(CONFIG["SAMPLE_ROWS"]).collect()
        else:
            df = pl.read_parquet(path)
    else:
        raise ValueError(f"지원하지 않는 파일 형식입니다: {suffix}")

    # 원본 순서 보존. timestamp 동률일 때 안정적인 정렬 기준으로 사용.
    df = df.with_row_index("source_row_number")
    return df


df_raw = read_input_data(CONFIG["RAW_DATA_PATH"])
profile_memory(df_raw, "01_loaded_raw")

print(df_raw.columns)
df_raw.head(3)


[01_loaded_raw] rows=5,078,345, cols=12, memory=464.71 MB
['source_row_number', 'Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']


source_row_number,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
u32,str,str,str,str,str,f64,str,f64,str,str,i64
0,"""2022/09/01 00:20""","""010""","""8000EBD30""","""010""","""8000EBD30""",3697.34,"""US Dollar""",3697.34,"""US Dollar""","""Reinvestment""",0
1,"""2022/09/01 00:20""","""03208""","""8000F4580""","""001""","""8000F5340""",0.01,"""US Dollar""",0.01,"""US Dollar""","""Cheque""",0
2,"""2022/09/01 00:00""","""03209""","""8000F4670""","""03209""","""8000F4670""",14675.57,"""US Dollar""",14675.57,"""US Dollar""","""Reinvestment""",0


## 5. 컬럼명 표준화와 필수 컬럼 확인
ML/GNN 단계에서 같은 이름으로 컬럼을 사용할 수 있도록 sender/receiver 기준 컬럼을 정리합니다. 원본 컬럼은 검증 가능성을 위해 가능한 한 보존합니다.

In [5]:
# =========================
# 컬럼 탐지
# =========================

# 원본 컬럼명이 달라도 표준 컬럼으로 연결될 수 있는지 확인합니다.
def validate_required_columns(df):
    cols = df.columns
    detected = {
        "timestamp_col": detect_column(cols, CONFIG["TIMESTAMP_COL_CANDIDATES"], True, "timestamp"),
        "label_col": detect_column(cols, CONFIG["LABEL_COL_CANDIDATES"], True, "label"),
        "sender_col": detect_column(cols, CONFIG["SENDER_COL_CANDIDATES"], True, "sender account"),
        "receiver_col": detect_column(cols, CONFIG["RECEIVER_COL_CANDIDATES"], True, "receiver account"),
        "amount_col": detect_column(cols, CONFIG["AMOUNT_COL_CANDIDATES"], True, "amount"),
        "from_bank_col": detect_column(cols, CONFIG["FROM_BANK_COL_CANDIDATES"], False, "from bank"),
        "to_bank_col": detect_column(cols, CONFIG["TO_BANK_COL_CANDIDATES"], False, "to bank"),
        "payment_format_col": detect_column(cols, CONFIG["PAYMENT_FORMAT_COL_CANDIDATES"], False, "payment format"),
        "tx_id_col": detect_column(cols, CONFIG["TX_ID_COL_CANDIDATES"], False, "tx_id"),
    }

    amount_keys = {normalize_name(c) for c in CONFIG["AMOUNT_COL_CANDIDATES"]}
    currency_keys = {normalize_name(c) for c in CONFIG["CURRENCY_COL_CANDIDATES"]}
    detected["amount_cols_all"] = [c for c in cols if normalize_name(c) in amount_keys]
    detected["currency_cols_all"] = [c for c in cols if normalize_name(c) in currency_keys]

    if CONFIG["USE_BANK_ACCOUNT_COMPOSITE_ID"]:
        if detected["from_bank_col"] is None or detected["to_bank_col"] is None:
            raise ValueError("composite 계좌 ID를 만들려면 From Bank / To Bank 컬럼이 필요합니다.")

    add_validation("required_columns", "PASS", "P0", json.dumps(detected, ensure_ascii=False), 0)
    return detected


COLS = validate_required_columns(df_raw)
COLS


{'timestamp_col': 'Timestamp',
 'label_col': 'Is Laundering',
 'sender_col': 'Account',
 'receiver_col': 'Account.1',
 'amount_col': 'Amount Paid',
 'from_bank_col': 'From Bank',
 'to_bank_col': 'To Bank',
 'payment_format_col': 'Payment Format',
 'tx_id_col': None,
 'amount_cols_all': ['Amount Received', 'Amount Paid'],
 'currency_cols_all': ['Receiving Currency', 'Payment Currency']}

## 6. Timestamp 변환 및 검증
시간순 split과 누수 방지를 위해 Timestamp는 가장 중요한 기준 컬럼입니다. 파싱 실패가 있으면 몇 행이 문제인지 리포트에 남깁니다.

In [6]:
# =========================
# schema review
# =========================

# 변환 전후 컬럼 타입과 결측 상태를 비교할 수 있도록 리포트 행을 만듭니다.
def make_schema_review(df, detected_cols):
    n_rows = df.height
    check_cols = {
        "source_row_number",
        detected_cols["timestamp_col"],
        detected_cols["label_col"],
        detected_cols["sender_col"],
        detected_cols["receiver_col"],
        detected_cols["amount_col"],
    }
    for k in ["from_bank_col", "to_bank_col", "tx_id_col", "payment_format_col"]:
        if detected_cols.get(k):
            check_cols.add(detected_cols[k])
    check_cols.update(detected_cols.get("amount_cols_all", []))
    check_cols.update(detected_cols.get("currency_cols_all", []))

    rows = []
    for col in df.columns:
        null_count = df.select(pl.col(col).null_count()).item()
        unique_count = None
        if col in check_cols:
            try:
                unique_count = df.select(pl.col(col).n_unique()).item()
            except Exception:
                unique_count = None
        rows.append({
            "record_type": "schema",
            "column_name": col,
            "dtype": str(df.schema[col]),
            "null_count": int(null_count),
            "null_ratio": float(null_count / n_rows) if n_rows else None,
            "unique_count": unique_count,
            "check_name": None,
            "status": None,
            "severity": None,
            "details": None,
            "n_failed_rows": None,
            "created_at": _now(),
        })
    return pl.DataFrame(rows)


schema_review_before = make_schema_review(df_raw, COLS)
schema_review_before.head(20)


record_type,column_name,dtype,null_count,null_ratio,unique_count,check_name,status,severity,details,n_failed_rows,created_at
str,str,str,i64,f64,i64,null,null,null,null,null,str
"""schema""","""source_row_number""","""UInt32""",0,0.0,5078345,null,null,null,null,null,"""2026-05-22 07:56:48"""
"""schema""","""Timestamp""","""String""",0,0.0,15018,null,null,null,null,null,"""2026-05-22 07:56:48"""
"""schema""","""From Bank""","""String""",0,0.0,30528,null,null,null,null,null,"""2026-05-22 07:56:48"""
"""schema""","""Account""","""String""",0,0.0,496995,null,null,null,null,null,"""2026-05-22 07:56:49"""
"""schema""","""To Bank""","""String""",0,0.0,15850,null,null,null,null,null,"""2026-05-22 07:56:49"""
…,…,…,…,…,…,…,…,…,…,…,…
"""schema""","""Receiving Currency""","""String""",0,0.0,15,null,null,null,null,null,"""2026-05-22 07:56:50"""
"""schema""","""Amount Paid""","""Float64""",0,0.0,923873,null,null,null,null,null,"""2026-05-22 07:56:50"""
"""schema""","""Payment Currency""","""String""",0,0.0,15,null,null,null,null,null,"""2026-05-22 07:56:50"""


## 7. Label 검증
`Is_Laundering`은 예측 타겟이므로 0/1 이외 값이나 결측치를 임의로 0으로 채우지 않습니다. 문제가 있으면 기본적으로 중단하는 것이 안전합니다.

In [7]:
# =========================
# 핵심 컬럼 정리
# =========================

# 여러 timestamp 포맷을 순서대로 시도해서 파싱 성공률을 높입니다.
def parse_timestamp_expr(col_name):
    raw = pl.col(col_name).cast(pl.Utf8).str.strip_chars()
    parsed = [raw.str.strptime(pl.Datetime, format=fmt, strict=False) for fmt in CONFIG["TIMESTAMP_FORMATS"]]
    return pl.coalesce(parsed)


def account_id_expr(bank_col, account_col):
    bank = pl.col(bank_col).cast(pl.Utf8).str.strip_chars()
    acct = pl.col(account_col).cast(pl.Utf8).str.strip_chars()
    return pl.concat_str([bank, pl.lit("|"), acct])


# 원본 컬럼을 프로젝트 표준 컬럼명으로 정리합니다.
def cast_core_columns(df, detected):
    ts_col = detected["timestamp_col"]
    label_col = detected["label_col"]
    sender_col = detected["sender_col"]
    receiver_col = detected["receiver_col"]
    amount_cols = detected.get("amount_cols_all") or [detected["amount_col"]]

    exprs = [
        pl.col(ts_col).cast(pl.Utf8).str.strip_chars().alias("timestamp_raw"),
        parse_timestamp_expr(ts_col).alias("timestamp"),
        pl.col(label_col).cast(pl.Int64, strict=False).alias("is_laundering"),
    ]

    if CONFIG["USE_BANK_ACCOUNT_COMPOSITE_ID"]:
        exprs.extend([
            account_id_expr(detected["from_bank_col"], sender_col).alias("sender_account_id"),
            account_id_expr(detected["to_bank_col"], receiver_col).alias("receiver_account_id"),
            pl.col(detected["from_bank_col"]).cast(pl.Utf8).str.strip_chars().alias("sender_bank_id"),
            pl.col(detected["to_bank_col"]).cast(pl.Utf8).str.strip_chars().alias("receiver_bank_id"),
        ])
    else:
        exprs.extend([
            pl.col(sender_col).cast(pl.Utf8).str.strip_chars().alias("sender_account_id"),
            pl.col(receiver_col).cast(pl.Utf8).str.strip_chars().alias("receiver_account_id"),
        ])

    for col in amount_cols:
        exprs.append(pl.col(col).cast(pl.Float64, strict=False).alias(compact_col_name(col)))

    exprs.append(pl.col(detected["amount_col"]).cast(pl.Float64, strict=False).alias("amount"))

    out = df.with_columns(exprs)
    out = out.with_columns(pl.col("is_laundering").cast(pl.Int8, strict=False))
    return out


df = cast_core_columns(df_raw, COLS)
profile_memory(df, "02_core_columns")

df.select([
    "source_row_number", "timestamp_raw", "timestamp",
    "sender_account_id", "receiver_account_id", "amount", "is_laundering"
]).head(5)


[02_core_columns] rows=5,078,345, cols=22, memory=910.97 MB


source_row_number,timestamp_raw,timestamp,sender_account_id,receiver_account_id,amount,is_laundering
u32,str,datetime[μs],str,str,f64,i8
0,"""2022/09/01 00:20""",2022-09-01 00:20:00,"""010|8000EBD30""","""010|8000EBD30""",3697.34,0
1,"""2022/09/01 00:20""",2022-09-01 00:20:00,"""03208|8000F4580""","""001|8000F5340""",0.01,0
2,"""2022/09/01 00:00""",2022-09-01 00:00:00,"""03209|8000F4670""","""03209|8000F4670""",14675.57,0
3,"""2022/09/01 00:02""",2022-09-01 00:02:00,"""012|8000F5030""","""012|8000F5030""",2806.97,0
4,"""2022/09/01 00:06""",2022-09-01 00:06:00,"""010|8000F5200""","""010|8000F5200""",36682.97,0


## 8. 거래 ID와 계좌 ID 정리
거래 단위 예측을 위해 `tx_id`를 만들고, 송신/수신 계좌를 일관된 account id로 정리합니다. 이 매핑은 이후 GNN에서도 사용됩니다.

In [8]:
# =========================
# timestamp 검증 및 정렬
# =========================

# timestamp 파싱 실패 행을 확인하고 실패가 있으면 중단합니다.
def validate_timestamp_parse(df):
    total = df.height
    raw_null = df.select(pl.col("timestamp_raw").is_null().sum()).item()
    parsed_null = df.select(pl.col("timestamp").is_null().sum()).item()

    if parsed_null == total:
        add_validation("timestamp_parse", "FAIL", "P0", "all timestamp values failed to parse", parsed_null)
        raise ValueError("timestamp parsing이 전부 실패했습니다. TIMESTAMP_FORMATS를 확인하세요.")

    status = "PASS" if parsed_null == 0 else "WARN"
    add_validation(
        "timestamp_parse", status, "P0" if parsed_null else "INFO",
        f"raw_null={raw_null}, parsed_null={parsed_null}, total={total}",
        parsed_null,
    )
    print(f"timestamp parsed_null={parsed_null:,} / total={total:,}")
    return parsed_null


bad_ts = validate_timestamp_parse(df)
if bad_ts > 0:
    print(f"timestamp가 없는 row {bad_ts:,}건 제외")
    df = df.filter(pl.col("timestamp").is_not_null())

df = df.sort(["timestamp", "source_row_number"])

print("timestamp min:", df.select(pl.col("timestamp").min()).item())
print("timestamp max:", df.select(pl.col("timestamp").max()).item())
profile_memory(df, "03_timestamp_sorted")


timestamp parsed_null=0 / total=5,078,345
timestamp min: 2022-09-01 00:00:00
timestamp max: 2022-09-18 16:18:00
[03_timestamp_sorted] rows=5,078,345, cols=22, memory=910.36 MB


## 8-1. Primary period 기준 tail 제외

primary period 기준으로 원본 파일은 수정하지 않은 상태에서 `timestamp` cutoff로만 tail 구간을 제외.
primary period 밖 tail 데이터가 test label 분포 왜곡 방지.

In [9]:
# =========================
# primary period filter
# =========================

# 원본 CSV는 절대 수정하지 않습니다.
# 이 단계에서는 Kaggle primary period 밖 tail 구간만 timestamp 기준으로 제외합니다.
def parse_config_datetime(value, config_name):
    if value in [None, "", "None"]:
        return None
    try:
        return datetime.fromisoformat(str(value))
    except ValueError as e:
        raise ValueError(f"{config_name} 값이 ISO datetime 형식이 아닙니다: {value}") from e


def safe_stage_summary(df_stage, stage, note):
    row_count = df_stage.height
    if row_count == 0:
        positive_count = 0
        timestamp_min = None
        timestamp_max = None
    else:
        positive_count = int(
            df_stage.select(
                pl.col("is_laundering")
                .fill_null(0)
                .cast(pl.Int64, strict=False)
                .sum()
            ).item()
        )
        timestamp_min = df_stage.select(pl.col("timestamp").min()).item()
        timestamp_max = df_stage.select(pl.col("timestamp").max()).item()

    negative_count = int(row_count - positive_count)
    positive_rate = float(positive_count / row_count) if row_count else 0.0
    return {
        "stage": stage,
        "row_count": int(row_count),
        "positive_count": int(positive_count),
        "negative_count": negative_count,
        "positive_rate": positive_rate,
        "timestamp_min": timestamp_min,
        "timestamp_max": timestamp_max,
        "note": note,
        "created_at": _now(),
    }


def pct(x):
    if x is None:
        return "NA"
    return f"{float(x) * 100:.4f}%"


def apply_primary_period_filter(df):
    if not CONFIG.get("APPLY_PRIMARY_PERIOD_FILTER", True):
        add_validation(
            "primary_period_filter",
            "WARN",
            "P1",
            "APPLY_PRIMARY_PERIOD_FILTER=False. No primary period filtering applied.",
            0,
        )
        print("[PRIMARY PERIOD FILTER] skipped")
        return df

    start_dt = parse_config_datetime(
        CONFIG.get("PRIMARY_PERIOD_START_INCLUSIVE"),
        "PRIMARY_PERIOD_START_INCLUSIVE",
    )
    end_exclusive_dt = parse_config_datetime(
        CONFIG.get("PRIMARY_PERIOD_END_EXCLUSIVE"),
        "PRIMARY_PERIOD_END_EXCLUSIVE",
    )

    if start_dt is None and end_exclusive_dt is None:
        raise ValueError(
            "Primary period filter가 켜져 있지만 start/end cutoff가 모두 비어 있습니다. "
            "PRIMARY_PERIOD_START_INCLUSIVE 또는 PRIMARY_PERIOD_END_EXCLUSIVE를 설정하세요."
        )

    original_df = df
    keep_expr = pl.lit(True)
    note_parts = []

    if start_dt is not None:
        keep_expr = keep_expr & (pl.col("timestamp") >= pl.lit(start_dt))
        note_parts.append(f"timestamp >= {start_dt}")

    if end_exclusive_dt is not None:
        keep_expr = keep_expr & (pl.col("timestamp") < pl.lit(end_exclusive_dt))
        note_parts.append(f"timestamp < {end_exclusive_dt}")

    kept_df = original_df.filter(keep_expr)

    if end_exclusive_dt is not None:
        tail_df = original_df.filter(pl.col("timestamp") >= pl.lit(end_exclusive_dt))
    else:
        tail_df = original_df.filter(~keep_expr)

    report_rows = [
        safe_stage_summary(
            original_df,
            "original_raw",
            "Rows after timestamp parse validation and timestamp sort. Original source file is not modified.",
        ),
        safe_stage_summary(
            kept_df,
            "primary_period_kept",
            "Kept rows using primary period condition: " + " and ".join(note_parts),
        ),
        safe_stage_summary(
            tail_df,
            "tail_excluded",
            "Rows excluded after primary period end cutoff. This is timestamp-based, not row-count-based.",
        ),
    ]

    report = pl.DataFrame(report_rows)
    report_path = CONFIG["VALIDATION_DIR"] / "primary_period_filter_report.csv"
    write_csv(report, report_path)

    original_summary = report_rows[0]
    kept_summary = report_rows[1]
    tail_summary = report_rows[2]

    print("[PRIMARY PERIOD FILTER]")
    print("primary period start inclusive:", start_dt)
    print("primary period end exclusive cutoff:", end_exclusive_dt)
    print("original rows:", f"{original_summary['row_count']:,}")
    print("kept rows:", f"{kept_summary['row_count']:,}")
    print("excluded tail rows:", f"{tail_summary['row_count']:,}")
    print("original timestamp min/max:", original_summary["timestamp_min"], "/", original_summary["timestamp_max"])
    print("kept timestamp min/max:", kept_summary["timestamp_min"], "/", kept_summary["timestamp_max"])
    print("excluded timestamp min/max:", tail_summary["timestamp_min"], "/", tail_summary["timestamp_max"])
    print("original positive rate:", pct(original_summary["positive_rate"]))
    print("kept positive rate:", pct(kept_summary["positive_rate"]))
    print("excluded tail positive rate:", pct(tail_summary["positive_rate"]))
    print("saved:", report_path)

    if kept_df.height == 0:
        add_validation("primary_period_filter", "FAIL", "P0", "primary period kept rows = 0", original_df.height)
        raise ValueError("primary period 필터 결과 row가 0건입니다. cutoff 설정을 확인하세요.")

    excluded_rows = original_df.height - kept_df.height
    add_validation(
        "primary_period_filter",
        "PASS",
        "P0",
        (
            f"start={start_dt}, end_exclusive={end_exclusive_dt}, "
            f"original={original_df.height}, kept={kept_df.height}, excluded={excluded_rows}"
        ),
        excluded_rows,
    )

    # 필터 후에도 timestamp/source_row_number 기준 정렬을 다시 확정합니다.
    return kept_df.sort(["timestamp", "source_row_number"])


df = apply_primary_period_filter(df)
profile_memory(df, "03_1_primary_period_filtered")


[PRIMARY PERIOD FILTER]
primary period start inclusive: 2022-09-01 00:00:00
primary period end exclusive cutoff: 2022-09-11 00:00:00
original rows: 5,078,345
kept rows: 5,077,237
excluded tail rows: 1,108
original timestamp min/max: 2022-09-01 00:00:00 / 2022-09-18 16:18:00
kept timestamp min/max: 2022-09-01 00:00:00 / 2022-09-10 23:59:00
excluded timestamp min/max: 2022-09-11 00:05:00 / 2022-09-18 16:18:00
original positive rate: 0.1019%
kept positive rate: 0.0891%
excluded tail positive rate: 59.1155%
saved: /content/drive/MyDrive/Graph_AML/data/processed/clean_base/validation/primary_period_filter_report.csv
[03_1_primary_period_filtered] rows=5,077,237, cols=22, memory=910.17 MB


## 8-2. Primary period 기준 time-based 60/20/20 split 재생성

primary period만 남긴 뒤 timestamp 정렬 상태에서 60/20/20 train/validation/test split을 다시 만듭니다.  
`split` 컬럼은 clean_base에 포함되어 이후 ML/GNN이 동일한 split을 재사용.

In [10]:
# =========================
# time-based 60/20/20 split after primary period filter
# =========================

SPLIT_ORDER = ["train", "val", "test"]


def create_time_based_split(df):
    ratios = CONFIG["SPLIT_RATIOS"]
    ratio_sum = float(sum(ratios.values()))
    if abs(ratio_sum - 1.0) > 1e-9:
        raise ValueError(f"SPLIT_RATIOS 합이 1.0이 아닙니다: {ratio_sum}")

    n_rows = df.height
    n_train = int(n_rows * ratios["train"])
    n_val_end = int(n_rows * (ratios["train"] + ratios["val"]))

    if n_train <= 0 or n_val_end <= n_train or n_val_end >= n_rows:
        raise ValueError(
            f"split 경계가 비정상입니다. n_rows={n_rows}, n_train={n_train}, n_val_end={n_val_end}"
        )

    out = (
        df.sort(["timestamp", "source_row_number"])
        .with_row_index("_split_row_idx")
        .with_columns(
            pl.when(pl.col("_split_row_idx") < n_train)
            .then(pl.lit("train"))
            .when(pl.col("_split_row_idx") < n_val_end)
            .then(pl.lit("val"))
            .otherwise(pl.lit("test"))
            .alias("split")
        )
        .drop("_split_row_idx")
    )
    return out


def summarize_split(df):
    expected_rates = CONFIG.get("EXPECTED_SPLIT_POSITIVE_RATE", {})
    tolerance = float(CONFIG.get("EXPECTED_RATE_TOLERANCE", 0.00005))
    rows = []

    for split_name in SPLIT_ORDER:
        part = df.filter(pl.col("split") == split_name)
        row_count = part.height
        positive_count = int(part.select(pl.col("is_laundering").cast(pl.Int64).sum()).item()) if row_count else 0
        negative_count = int(row_count - positive_count)
        positive_rate = float(positive_count / row_count) if row_count else 0.0
        expected_rate = expected_rates.get(split_name)
        diff = None if expected_rate is None else float(positive_rate - expected_rate)
        rate_check = "NA"
        if expected_rate is not None:
            rate_check = "PASS" if abs(diff) <= tolerance else "REVIEW"

        rows.append({
            "split": split_name,
            "row_count": int(row_count),
            "positive_count": int(positive_count),
            "negative_count": negative_count,
            "positive_rate": positive_rate,
            "positive_rate_pct": positive_rate * 100,
            "expected_positive_rate": expected_rate,
            "expected_positive_rate_pct": None if expected_rate is None else expected_rate * 100,
            "diff_from_expected": diff,
            "diff_from_expected_pct_point": None if diff is None else diff * 100,
            "timestamp_min": part.select(pl.col("timestamp").min()).item() if row_count else None,
            "timestamp_max": part.select(pl.col("timestamp").max()).item() if row_count else None,
            "rate_check": rate_check,
            "created_at": _now(),
        })

    return pl.DataFrame(rows)


def save_and_print_split_summary(df):
    split_summary = summarize_split(df)
    path = CONFIG["VALIDATION_DIR"] / "split_summary.csv"
    write_csv(split_summary, path)

    print("[SPLIT SUMMARY AFTER PRIMARY PERIOD FILTER]")
    for r in split_summary.to_dicts():
        print(
            f"{r['split']} rows / positives / positive rate: "
            f"{r['row_count']:,} / {r['positive_count']:,} / {r['positive_rate_pct']:.4f}% "
            f"(expected={r['expected_positive_rate_pct']:.4f}%, "
            f"diff={r['diff_from_expected_pct_point']:.6f}%p, check={r['rate_check']})"
        )
    print("saved:", path)

    review_rows = split_summary.filter(pl.col("rate_check") == "REVIEW").height
    if review_rows:
        add_validation(
            "split_expected_label_rate",
            "WARN",
            "P1",
            f"{review_rows} split(s) differ from expected rates beyond tolerance. Check split_summary.csv.",
            review_rows,
        )
    else:
        add_validation(
            "split_expected_label_rate",
            "PASS",
            "P1",
            "train/val/test positive rates are close to expected primary-period rates",
            0,
        )

    split_counts = {r["split"]: r["row_count"] for r in split_summary.to_dicts()}
    add_validation("time_based_split", "PASS", "P0", json.dumps(split_counts, ensure_ascii=False), 0)
    return split_summary


# primary period 기준으로 split을 다시 확정합니다.
df = create_time_based_split(df)
split_summary = save_and_print_split_summary(df)
profile_memory(df, "03_2_time_based_split_created")

split_summary


[SPLIT SUMMARY AFTER PRIMARY PERIOD FILTER]
train rows / positives / positive rate: 3,046,342 / 2,297 / 0.0754% (expected=0.0754%, diff=0.000002%p, check=PASS)
val rows / positives / positive rate: 1,015,447 / 1,082 / 0.1066% (expected=0.1066%, diff=-0.000046%p, check=PASS)
test rows / positives / positive rate: 1,015,448 / 1,143 / 0.1126% (expected=0.1126%, diff=-0.000039%p, check=PASS)
saved: /content/drive/MyDrive/Graph_AML/data/processed/clean_base/validation/split_summary.csv
[03_2_time_based_split_created] rows=5,077,237, cols=23, memory=931.48 MB


split,row_count,positive_count,negative_count,positive_rate,positive_rate_pct,expected_positive_rate,expected_positive_rate_pct,diff_from_expected,diff_from_expected_pct_point,timestamp_min,timestamp_max,rate_check,created_at
str,i64,i64,i64,f64,f64,f64,f64,f64,f64,datetime[μs],datetime[μs],str,str
"""train""",3046342,2297,3044045,0.000754,0.075402,0.000754,0.0754,1.9083e-8,0.000002,2022-09-01 00:00:00,2022-09-06 13:34:00,"""PASS""","""2026-05-22 07:57:01"""
"""val""",1015447,1082,1014365,0.001066,0.106554,0.001066,0.1066,-4.5941e-7,-0.000046,2022-09-06 13:34:00,2022-09-08 16:09:00,"""PASS""","""2026-05-22 07:57:01"""
"""test""",1015448,1143,1014305,0.001126,0.112561,0.001126,0.1126,-3.8845e-7,-0.000039,2022-09-08 16:09:00,2022-09-10 23:59:00,"""PASS""","""2026-05-22 07:57:01"""


## 9. 공통 clean base 생성
ML/GNN이 공통으로 사용할 기준 테이블을 만듭니다. 이 파일에는 아직 ML history feature나 PageRank 같은 파생 피처를 넣지 않았습니다.

In [11]:
# =========================
# label 검증
# =========================

# 라벨이 0/1로만 구성되어 있는지 검증합니다.
def validate_label_binary(df, strict=True):
    total = df.height
    invalid = df.filter(
        pl.col("is_laundering").is_null() |
        (~pl.col("is_laundering").is_in([0, 1]))
    )
    invalid_count = invalid.height
    invalid_ratio = invalid_count / total if total else 0.0

    rows = [
        {"report_type": "summary", "metric": "total_rows", "value": total, "details": None},
        {"report_type": "summary", "metric": "invalid_label_rows", "value": invalid_count, "details": f"ratio={invalid_ratio:.8f}"},
        {"report_type": "summary", "metric": "strict_label_check", "value": int(strict), "details": None},
    ]

    dist = df.group_by("is_laundering").agg(pl.count().alias("row_count")).sort("is_laundering")
    for r in dist.to_dicts():
        rows.append({
            "report_type": "label_distribution",
            "metric": f"is_laundering={r['is_laundering']}",
            "value": r["row_count"],
            "details": None,
        })

    if invalid_count > 0:
        keep_cols = [
            "source_row_number", "timestamp_raw", "timestamp",
            "sender_account_id", "receiver_account_id", "amount", "is_laundering"
        ]
        for i, r in enumerate(invalid.select([c for c in keep_cols if c in invalid.columns]).head(20).to_dicts()):
            rows.append({
                "report_type": "invalid_example",
                "metric": f"example_{i}",
                "value": None,
                "details": json.dumps(r, ensure_ascii=False, default=str),
            })

    report = pl.DataFrame(rows)
    path = CONFIG["VALIDATION_DIR"] / "label_verification_report.csv"
    write_csv(report, path)
    print("saved:", path)

    if invalid_count > 0:
        add_validation("label_binary", "FAIL", "P0", f"invalid_count={invalid_count}", invalid_count)
        if strict:
            raise ValueError("label 값이 0/1이 아닌 row가 있습니다. label_verification_report.csv를 확인하세요.")
        print(f"label 오류 row {invalid_count:,}건 제외")
        return df.filter(pl.col("is_laundering").is_in([0, 1]))

    add_validation("label_binary", "PASS", "P0", "all labels are 0/1", 0)
    return df


df = validate_label_binary(df, CONFIG["STRICT_LABEL_CHECK"])
profile_memory(df, "04_label_checked")

df.group_by("is_laundering").agg(pl.count().alias("row_count")).sort("is_laundering")

saved: /content/drive/MyDrive/Graph_AML/data/processed/clean_base/validation/label_verification_report.csv
[04_label_checked] rows=5,077,237, cols=23, memory=931.48 MB


is_laundering,row_count
i8,u32
0,5072715
1,4522


## 10. account_mapping 생성
계좌 문자열을 모델과 그래프에서 쓰기 쉬운 정수형 ID로 매핑합니다. GNN 단계에서 새로 만들지 않고 이 매핑을 사용하기 위함입니다.

In [12]:
# =========================
# tx_id 생성
# =========================

# 원본 거래 ID가 없을 때도 재현 가능한 tx_id를 생성합니다.
def create_tx_id(df, detected):
    tx_col = detected.get("tx_id_col")
    if tx_col:
        null_cnt = df.select(pl.col(tx_col).is_null().sum()).item()
        uniq_cnt = df.select(pl.col(tx_col).n_unique()).item()
        dup_cnt = df.height - uniq_cnt
        if null_cnt == 0 and dup_cnt == 0:
            print(f"기존 거래 ID 사용: {tx_col}")
            return df.with_columns(pl.col(tx_col).cast(pl.Utf8).alias("tx_id"))
        print(f"기존 거래 ID 사용 안 함: null={null_cnt}, dup={dup_cnt}")

    # 정렬 이후 순번. 이후 파일 병합 기준으로만 사용.
    return df.with_row_index("tx_id")


def validate_tx_id_unique(df):
    null_cnt = df.select(pl.col("tx_id").is_null().sum()).item()
    uniq_cnt = df.select(pl.col("tx_id").n_unique()).item()
    dup_cnt = df.height - uniq_cnt

    if null_cnt or dup_cnt:
        add_validation("tx_id_unique", "FAIL", "P0", f"null={null_cnt}, dup={dup_cnt}", null_cnt + dup_cnt)
        raise AssertionError("tx_id에 null 또는 중복이 있습니다.")

    add_validation("tx_id_unique", "PASS", "P0", f"unique={uniq_cnt}", 0)
    print("tx_id check: PASS")


# 이미 tx_id가 있으면 중복 생성 방지
if "tx_id" in df.columns:
    df = df.drop("tx_id")

# 이후 ML/GNN 산출물 연결을 위해 tx_id를 확정합니다.
df = create_tx_id(df, COLS)
validate_tx_id_unique(df)

df.select(["tx_id", "source_row_number", "timestamp", "sender_account_id", "receiver_account_id"]).head(5)


tx_id check: PASS


tx_id,source_row_number,timestamp,sender_account_id,receiver_account_id
u32,u32,datetime[μs],str,str
0,2,2022-09-01 00:00:00,"""03209|8000F4670""","""03209|8000F4670"""
1,17,2022-09-01 00:00:00,"""01420|8005DFEB0""","""01420|8005DFEB0"""
2,158,2022-09-01 00:00:00,"""010|8000F6850""","""010|8000F6850"""
3,218,2022-09-01 00:00:00,"""012|8001167D0""","""012|8001167D0"""
4,281,2022-09-01 00:00:00,"""001|8001364A0""","""001|8001364A0"""


## 11. 스키마/라벨/메모리 리포트 저장
데이터 타입, 삭제 row, 라벨 검증, 메모리 사용량을 리포트로 남깁니다.

In [13]:
# =========================
# account_mapping 생성
# =========================

# sender/receiver 계좌를 하나의 노드 ID 체계로 매핑합니다.
def create_account_mapping(df):
    sender_null = df.select(pl.col("sender_account_id").is_null().sum()).item()
    receiver_null = df.select(pl.col("receiver_account_id").is_null().sum()).item()

    if sender_null or receiver_null:
        add_validation("account_id_null", "FAIL", "P0", f"sender_null={sender_null}, receiver_null={receiver_null}", sender_null + receiver_null)
        raise ValueError("sender/receiver 계좌 ID에 null이 있습니다.")

    accounts = pl.concat([
        df.select(pl.col("sender_account_id").alias("account_id")),
        df.select(pl.col("receiver_account_id").alias("account_id")),
    ])

    mapping = (
        accounts
        .unique()
        .sort("account_id")
        .with_row_index("node_idx")
        .select(["account_id", "node_idx"])
    )

    path = CONFIG["OUTPUT_DIR"] / "account_mapping.csv"
    write_csv(mapping, path)
    print("saved:", path)
    return mapping


def validate_account_mapping(df, mapping):
    null_cnt = mapping.select(pl.col("account_id").is_null().sum()).item()
    dup_cnt = mapping.height - mapping.select(pl.col("account_id").n_unique()).item()
    if null_cnt or dup_cnt:
        add_validation("account_mapping_unique", "FAIL", "P0", f"null={null_cnt}, dup={dup_cnt}", null_cnt + dup_cnt)
        raise AssertionError("account_mapping에 null 또는 중복이 있습니다.")

    sender_missing = (
        df.select(pl.col("sender_account_id").alias("account_id"))
        .join(mapping, on="account_id", how="left")
        .filter(pl.col("node_idx").is_null())
        .height
    )
    receiver_missing = (
        df.select(pl.col("receiver_account_id").alias("account_id"))
        .join(mapping, on="account_id", how="left")
        .filter(pl.col("node_idx").is_null())
        .height
    )

    if sender_missing or receiver_missing:
        add_validation("account_mapping_coverage", "FAIL", "P0", f"sender_missing={sender_missing}, receiver_missing={receiver_missing}", sender_missing + receiver_missing)
        raise AssertionError("account_mapping 누락이 있습니다.")

    add_validation("account_mapping_coverage", "PASS", "P0", f"n_accounts={mapping.height}", 0)
    print(f"account_mapping check: PASS, accounts={mapping.height:,}")


# GNN과 ML 모두 같은 계좌 매핑을 쓰도록 여기서 만듭니다.
account_mapping = create_account_mapping(df)
validate_account_mapping(df, account_mapping)
account_mapping.head(5)


saved: /content/drive/MyDrive/Graph_AML/data/processed/clean_base/account_mapping.csv
account_mapping check: PASS, accounts=515,078


account_id,node_idx
str,u32
"""0010057|801FB1090""",0
"""0010057|803A115E0""",1
"""0010057|803AA8E90""",2
"""0010057|803AAB430""",3
"""0010057|803AACE20""",4


## 12. clean_base 저장

후속 ML/GNN 노트북이 동일한 기준 데이터를 쓰도록 `split` 컬럼이 포함된 clean_base를 저장합니다.

In [14]:
# =========================
# 최종 검증
# =========================

# 저장 전 필수 컬럼과 mapping 정합성을 마지막으로 점검합니다.
# 최종 저장 전에 필수 검증을 통과했는지 확인합니다.
def run_all_validations(df, account_mapping):
    required = [
        "tx_id", "source_row_number", "timestamp", "timestamp_raw",
        "sender_account_id", "receiver_account_id", "amount", "is_laundering", "split",
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        add_validation("clean_base_required_columns", "FAIL", "P0", f"missing={missing}", 0)
        raise AssertionError(f"clean_base 필수 컬럼 누락: {missing}")
    add_validation("clean_base_required_columns", "PASS", "P0", f"required={required}", 0)

    validate_timestamp_parse(df)
    validate_tx_id_unique(df)
    validate_account_mapping(df, account_mapping)

    bad_label = df.filter(
        pl.col("is_laundering").is_null() |
        (~pl.col("is_laundering").is_in([0, 1]))
    ).height
    if bad_label:
        add_validation("final_label_check", "FAIL", "P0", f"bad_label={bad_label}", bad_label)
        raise AssertionError("최종 데이터에 label 오류가 남아 있습니다.")
    add_validation("final_label_check", "PASS", "P0", "labels are 0/1", 0)

    split_values = set(df.select("split").unique().to_series().to_list())
    expected_split_values = {"train", "val", "test"}
    if split_values != expected_split_values:
        add_validation("final_split_check", "FAIL", "P0", f"split_values={sorted(split_values)}", 0)
        raise AssertionError(f"최종 split 값이 train/val/test와 다릅니다: {sorted(split_values)}")
    add_validation("final_split_check", "PASS", "P0", f"split_values={sorted(split_values)}", 0)

    print("final validation: PASS")


# 최종 저장 전에 필수 검증을 통과했는지 확인합니다.
run_all_validations(df, account_mapping)


timestamp parsed_null=0 / total=5,077,237
tx_id check: PASS
account_mapping check: PASS, accounts=515,078
final validation: PASS


## 13. 최종 확인 출력
생성된 파일과 주요 row 수를 화면에 출력합니다. 실행 후 이 셀을 보고 정상 완료 여부를 빠르게 확인하면 됩니다.

In [15]:
# =========================
# clean_base 저장
# =========================

front_cols = [
    "tx_id", "source_row_number", "split", "timestamp", "timestamp_raw",
    "sender_account_id", "receiver_account_id", "sender_bank_id", "receiver_bank_id",
    "amount", "is_laundering",
]
front_cols = [c for c in front_cols if c in df.columns]
other_cols = [c for c in df.columns if c not in front_cols]

df_clean = df.select(front_cols + other_cols)

clean_base_path = CONFIG["OUTPUT_DIR"] / "clean_base.parquet"
df_clean.write_parquet(clean_base_path)
print("saved:", clean_base_path)
profile_memory(df_clean, "05_clean_base_saved")

df_clean.head(5)


saved: /content/drive/MyDrive/Graph_AML/data/processed/clean_base/clean_base.parquet
[05_clean_base_saved] rows=5,077,237, cols=24, memory=950.84 MB


tx_id,source_row_number,split,timestamp,timestamp_raw,sender_account_id,receiver_account_id,sender_bank_id,receiver_bank_id,amount,is_laundering,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,amount_received,amount_paid
u32,u32,str,datetime[μs],str,str,str,str,str,f64,i8,str,str,str,str,str,f64,str,f64,str,str,i64,f64,f64
0,2,"""train""",2022-09-01 00:00:00,"""2022/09/01 00:00""","""03209|8000F4670""","""03209|8000F4670""","""03209""","""03209""",14675.57,0,"""2022/09/01 00:00""","""03209""","""8000F4670""","""03209""","""8000F4670""",14675.57,"""US Dollar""",14675.57,"""US Dollar""","""Reinvestment""",0,14675.57,14675.57
1,17,"""train""",2022-09-01 00:00:00,"""2022/09/01 00:00""","""01420|8005DFEB0""","""01420|8005DFEB0""","""01420""","""01420""",897.37,0,"""2022/09/01 00:00""","""01420""","""8005DFEB0""","""01420""","""8005DFEB0""",897.37,"""US Dollar""",897.37,"""US Dollar""","""Reinvestment""",0,897.37,897.37
2,158,"""train""",2022-09-01 00:00:00,"""2022/09/01 00:00""","""010|8000F6850""","""010|8000F6850""","""010""","""010""",99986.94,0,"""2022/09/01 00:00""","""010""","""8000F6850""","""010""","""8000F6850""",99986.94,"""US Dollar""",99986.94,"""US Dollar""","""Reinvestment""",0,99986.94,99986.94
3,218,"""train""",2022-09-01 00:00:00,"""2022/09/01 00:00""","""012|8001167D0""","""012|8001167D0""","""012""","""012""",16.08,0,"""2022/09/01 00:00""","""012""","""8001167D0""","""012""","""8001167D0""",16.08,"""US Dollar""",16.08,"""US Dollar""","""Reinvestment""",0,16.08,16.08
4,281,"""train""",2022-09-01 00:00:00,"""2022/09/01 00:00""","""001|8001364A0""","""001|8001364A0""","""001""","""001""",10.3,0,"""2022/09/01 00:00""","""001""","""8001364A0""","""001""","""8001364A0""",10.3,"""US Dollar""",10.3,"""US Dollar""","""Reinvestment""",0,10.3,10.3


In [16]:
# ============================================================
# 최종 확인
# ============================================================
# 이 단계의 산출물은 clean_base와 account_mapping만 확인합니다.
# label 검증표는 LOCAL_OUTPUT_DIR에 저장하고, clean_base/account_mapping은 Drive에 저장합니다.

outputs = {
    "clean_base.parquet": CONFIG["DRIVE_OUTPUT_DIR"] / "clean_base.parquet",
    "account_mapping.csv": CONFIG["DRIVE_OUTPUT_DIR"] / "account_mapping.csv",
    "split_summary.csv": CONFIG["VALIDATION_DIR"] / "split_summary.csv",
    "label_verification_report.csv": CONFIG["VALIDATION_DIR"] / "label_verification_report.csv",
    "primary_period_filter_report.csv": CONFIG["VALIDATION_DIR"] / "primary_period_filter_report.csv",
}

print("생성 파일")
for name, path in outputs.items():
    print(f"- {name}: {path} / exists={path.exists()}")

print("\nclean_base shape:", df_clean.shape)
print("account_mapping shape:", account_mapping.shape)

print("\ntimestamp range")
print(df_clean.select([
    pl.col("timestamp").min().alias("min"),
    pl.col("timestamp").max().alias("max"),
]))

print("\nlabel distribution")
print(
    df_clean.group_by("is_laundering")
    .agg(pl.len().alias("row_count"))
    .with_columns((pl.col("row_count") / pl.col("row_count").sum()).alias("ratio"))
    .sort("is_laundering")
)

print("\nsplit summary")
print(
    df_clean.group_by("split")
    .agg([
        pl.len().alias("row_count"),
        pl.col("is_laundering").sum().alias("positive_count"),
        pl.col("timestamp").min().alias("timestamp_min"),
        pl.col("timestamp").max().alias("timestamp_max"),
    ])
    .with_columns((pl.col("positive_count") / pl.col("row_count") * 100).alias("positive_rate_pct"))
    .sort("split")
)

print("\npreprocessing check: PASS")

생성 파일
- clean_base.parquet: /content/drive/MyDrive/Graph_AML/data/processed/clean_base/clean_base.parquet / exists=True
- account_mapping.csv: /content/drive/MyDrive/Graph_AML/data/processed/clean_base/account_mapping.csv / exists=True
- split_summary.csv: /content/drive/MyDrive/Graph_AML/data/processed/clean_base/validation/split_summary.csv / exists=True
- label_verification_report.csv: /content/drive/MyDrive/Graph_AML/data/processed/clean_base/validation/label_verification_report.csv / exists=True
- primary_period_filter_report.csv: /content/drive/MyDrive/Graph_AML/data/processed/clean_base/validation/primary_period_filter_report.csv / exists=True

clean_base shape: (5077237, 24)
account_mapping shape: (515078, 2)

timestamp range
shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ min                 ┆ max                 │
│ ---                 ┆ ---                 │
│ datetime[μs]        ┆ datetime[μs]        │
╞═════════════════════╪═════════════════════╡
│ 2022-09-01